<a href="https://colab.research.google.com/github/Nurana100/colab/blob/main/citi_market_data_monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!apt update -q
!apt-get install -q openjdk-11-jdk-headless


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
Building dependency tree...
Reading state information...
101 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state i

In [9]:
!java -version
!javac -version

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)
javac 17.0.19


In [10]:
from getpass import getpass
import os

os.environ["TWELVE_DATA_API_KEY"] = getpass("Enter your Twelve Data API key: ")

Enter your Twelve Data API key: ··········


In [11]:
%%writefile App.java
import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.net.HttpURLConnection;
import java.net.URL;
import java.time.LocalDateTime;
import java.util.LinkedList;
import java.util.Queue;

public class App {

    // Simple class to hold a price + timestamp together
    static class PricePoint {
        double price;
        LocalDateTime timestamp;

        PricePoint(double price, LocalDateTime timestamp) {
            this.price = price;
            this.timestamp = timestamp;
        }

        @Override
        public String toString() {
            return "price=" + price + ", timestamp=" + timestamp;
        }
    }

    public static void main(String[] args) throws Exception {
        String apiKey = System.getenv("TWELVE_DATA_API_KEY");
        if (apiKey == null || apiKey.isEmpty()) {
            System.out.println("Error: TWELVE_DATA_API_KEY environment variable not set.");
            return;
        }

        Queue<PricePoint> priceQueue = new LinkedList<>();

        // Run 5 times as a demo (every 15 seconds).
        // Change this loop condition to `while (true)` for continuous polling.
        int iterations = 5;

        for (int i = 0; i < iterations; i++) {
            try {
                double price = getDiaPrice(apiKey);
                LocalDateTime timestamp = LocalDateTime.now();

                PricePoint point = new PricePoint(price, timestamp);
                priceQueue.add(point);

                System.out.println("Added data point: " + point);
                System.out.println("Current queue size: " + priceQueue.size());

            } catch (Exception e) {
                System.out.println("Error retrieving price: " + e.getMessage());
            }

            // Wait 15 seconds before the next request
            Thread.sleep(15000);
        }
    }

    private static double getDiaPrice(String apiKey) throws Exception {
        String urlString = "https://api.twelvedata.com/price?symbol=DIA&apikey=" + apiKey;
        URL url = new URL(urlString);

        HttpURLConnection connection = (HttpURLConnection) url.openConnection();
        connection.setRequestMethod("GET");

        BufferedReader reader = new BufferedReader(
            new InputStreamReader(connection.getInputStream())
        );

        StringBuilder response = new StringBuilder();
        String line;
        while ((line = reader.readLine()) != null) {
            response.append(line);
        }
        reader.close();

        // Response looks like: {"price":"420.15"}
        String json = response.toString();
        String priceStr = json.replaceAll(".*\"price\":\"([^\"]+)\".*", "$1");

        return Double.parseDouble(priceStr);
    }
}

Overwriting App.java


In [12]:
import subprocess
import os

print("API key found:", bool(os.getenv("TWELVE_DATA_API_KEY")))

compile_result = subprocess.run(
    ["javac", "App.java"],
    capture_output=True,
    text=True
)

print("Compile return code:", compile_result.returncode)

if compile_result.stdout:
    print("Compile output:")
    print(compile_result.stdout)

if compile_result.stderr:
    print("Compile errors:")
    print(compile_result.stderr)

if compile_result.returncode == 0:
    run_result = subprocess.run(
        ["java", "App"],
        capture_output=True,
        text=True,
        env=os.environ
    )

    print("Run return code:", run_result.returncode)

    print("Application output:")
    print(run_result.stdout)

    if run_result.stderr:
        print("Application errors:")
        print(run_result.stderr)
else:
    print("Your Java file did not compile. Review the errors above before running the application.")

API key found: True
Compile return code: 0
Run return code: 0
Application output:
Added data point: price=526.01001, timestamp=2026-07-16T06:55:25.034107861
Current queue size: 1
Added data point: price=526.01001, timestamp=2026-07-16T06:55:40.225129595
Current queue size: 2
Added data point: price=526.01001, timestamp=2026-07-16T06:55:55.424291148
Current queue size: 3
Added data point: price=526.01001, timestamp=2026-07-16T06:56:10.604965909
Current queue size: 4
Added data point: price=526.01001, timestamp=2026-07-16T06:56:25.791083148
Current queue size: 5



In [15]:
%%writefile visualize_data.py
import re
import pandas as pd
import matplotlib.pyplot as plt

java_output = """
Added data point: price=526.01001, timestamp=2026-07-16T06:55:25.034107861
Current queue size: 1
Added data point: price=526.01001, timestamp=2026-07-16T06:55:40.225129595
Current queue size: 2
Added data point: price=526.01001, timestamp=2026-07-16T06:55:55.424291148
Current queue size: 3
Added data point: price=526.01001, timestamp=2026-07-16T06:56:10.604965909
Current queue size: 4
Added data point: price=526.01001, timestamp=2026-07-16T06:56:25.791083148
Current queue size: 5
"""

matches = re.findall(
    r"price=([0-9.]+), timestamp=([^\n]+)",
    java_output
)

df = pd.DataFrame(matches, columns=["price", "timestamp"])
df["price"] = df["price"].astype(float)
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(df)

plt.figure(figsize=(10, 5))
plt.plot(df["timestamp"], df["price"], marker="o")
plt.title("DIA Price Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Writing visualize_data.py


In [16]:
from google.colab import files
files.download('visualize_data.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>